#### MLflow Tracking Server API that automates the full lifecycle of an experiment

**Tracking URI Setup** → It begins by setting the MLflow tracking URI (`mlflow.set_tracking_uri(uri)`), which tells MLflow where to store metadata and artifacts. In a tracking server setup, this URI points to your backend store (e.g., SQLite, PostgreSQL) and artifact store (e.g., local folder, S3).

**Experiment Management** → Using the `MlflowClient`, it checks if the experiment already exists. If it’s marked as deleted, the function restores it (`client.restore_experiment`). If it doesn’t exist, it creates a new one. This ensures that experiments are reusable and consistent across multiple runs.

**Experiment Metadata** → It retrieves and prints experiment details from the tracking server (name, ID, artifact location, tags, lifecycle stage, creation time). This is exactly what the tracking server stores in its backend database.

**Run Lifecycle** → Inside `mlflow.start_run()`, the function starts a run under the chosen experiment. Runs are the atomic unit in MLflow tracking: they capture parameters, metrics, tags, and artifacts.

**Logging to Server →**

- Parameters (`mlflow.log_param`) → hyperparameters like C, l1_ratio, solver, penalty.
- Metrics (`mlflow.log_metric`) → accuracy, ROC‑AUC, MSE, MAE.
- Model (`mlflow.sklearn.log_model`) → serialized and stored in the artifact store.
- Artifacts (`mlflow.log_artifacts`) → additional files (e.g., data, plots).

**Artifact URI** → It prints the artifact path, which is managed by the tracking server and points to where files are stored (local folder or remote storage).

**Run Closure & Metadata** → After ending the run (`mlflow.end_run()`), it fetches the last active run (mlflow.last_active_run) and prints its run ID and run name. These identifiers are critical for querying results later in the MLflow UI or API.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc,precision_score,recall_score
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score,f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import argparse
import warnings
import mlflow
from mlflow.tracking import MlflowClient
import mlflow.sklearn
import mlflow.xgboost
import pickle
from pathlib import Path
# ---- Configure warnings and stdout ----
warnings.filterwarnings("ignore", category=UserWarning)
#sys.stdout.reconfigure(encoding='utf-8')
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

#### Get Cleaned Dataset with Feature Scaling

In [2]:
# Function: drop rows with non-numeric values
def drop_non_numeric(df_frame):
    cleaned_df = df_frame.copy()
    for col in cleaned_df.columns:
        # Try to convert column to numeric
        cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors='coerce')
    # Drop rows where conversion failed (NaN introduced)
    cleaned_df = cleaned_df.dropna()
    return cleaned_df

def categorize_chol(val):
    if val < 200:
        return "Normal", 0
    elif 200 <= val <= 239:
        return "Medium", 1
    else:
        return "High", 1
 
def get_cleaned_data():
    # Get current working directory
    cwd = os.getcwd()
    # Specify dataset filename
    filename = "dataset_2190_cholesterol.csv"
    file_path = os.path.join(cwd, filename)
    df = pd.read_csv(file_path)
    null_counts = df.isnull().sum()
    #if null_counts.sum() > 0:
    #    print("\n✅ Null values are present in the dataset.")
    #else:
    #    print("\n❌ No null values found in the dataset.")
    # Apply cleaning
    df_clean = drop_non_numeric(df)
    #print("Cleaned shape:", df_clean.shape)
    df_clean[['chol_category_label', 'chol_category_code']] = df_clean['chol'].apply(
        lambda x: pd.Series(categorize_chol(x))
        )
    # Define columns and target
    columns = ['age', 'sex', 'cp', 'trestbps', 'fbs', 'restecg', 'thalach', 'exang',
           'oldpeak', 'slope', 'ca', 'thal', 'num', 'chol_category_code']
    target = "chol_category_code"
    binary_vars = ['sex', 'fbs', 'exang']
    # Separate features and target
    X = df_clean[columns].drop(columns=[target])
    y = df_clean[target]
    
    # Identify numeric columns to scale (exclude binary + target)
    numeric_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col not in binary_vars]
    
    # Apply StandardScaler
    scaler = StandardScaler()
    X_scaled = X.copy()
    X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])
    
    # Store result in df_scale (features + target)
    df_scale = X_scaled.copy()
    df_scale[target] = y
    
    #print("Scaled dataset preview:")
    #print(df_scale.columns)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )
    #print("\nTrain set shape:", X_train.shape, y_train.shape)
    #print("Test set shape:", X_test.shape, y_test.shape)
    return X_train, X_test, y_train, y_test

In [3]:
def get_mlflow_uri_path() -> str:
    """
    Construct MLflow tracking URI path dynamically.
    Returns: file:/<current_working_directory>/mlruns
    """
    cwd = os.getcwd()   # current working directory
    uri_path = f"file:{os.path.join(cwd, 'mlruns')}"
    return uri_path

#### Models:

The logistic_model_1 API trains a logistic regression classifier using scikit‑learn with ElasticNet regularization (penalty='elasticnet', solver='saga', C=10, l1_ratio=0.1), fits it on the training data, and evaluates predictions on the test set by computing accuracy, ROC‑AUC, mean squared error, and mean absolute error. It returns both the trained model and a results DataFrame containing the hyperparameters and metrics, while also printing a summary of the experiment, making it a reusable component for ML pipelines and MLflow logging.

In [4]:
def logistic_model_1(X_train, X_test, y_train, y_test):
    # Logistic Regression with elasticnet penalty, saga solver
    # 1. saga + elasticnet + l1_ratio=0.1, C=10
    C_value = 10
    L_1_ratio = 0.1
    Solver = 'saga'
    Panality = 'elasticnet'
    model = LogisticRegression(
    penalty=Panality,
    solver=Solver,
    l1_ratio=L_1_ratio,
    C=C_value,
    max_iter=10000
    )
    model.fit(X_train, y_train)

    # Predictions
    y_pred_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1
    y_pred_class = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred_class)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    mse = mean_squared_error(y_test, y_pred_prob)
    mae = mean_absolute_error(y_test, y_pred_prob)

    # Store results in DataFrame
    results_df = pd.DataFrame([{
        "C": C_value,
        "l1_ratio": L_1_ratio,
        'solver': Solver,
        'penalty' : Panality,
        "Accuracy": acc,
        "ROC_AUC": roc_auc,
        "MSE": mse,
        "MAE": mae
    }])
    print("================ Logistice Model ================")
    print(f'Logistice accureacy {acc}')
    return model, results_df


In [5]:
def logistic_model_2(X_train, X_test, y_train, y_test):
    # Logistic Regression with elasticnet penalty, saga solver
    # 1. saga + elasticnet + l1_ratio=0.1, C=10
    C_value = 1
    L_1_ratio = 0.5
    Solver = 'saga'
    Panality = 'elasticnet'
    model = LogisticRegression(
    penalty=Panality,
    solver=Solver,
    l1_ratio=L_1_ratio,
    C=C_value,
    max_iter=10000
    )
    model.fit(X_train, y_train)

    # Predictions
    y_pred_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1
    y_pred_class = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred_class)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    mse = mean_squared_error(y_test, y_pred_prob)
    mae = mean_absolute_error(y_test, y_pred_prob)

    # Store results in DataFrame
    results_df = pd.DataFrame([{
        "C": C_value,
        "l1_ratio": L_1_ratio,
        'solver': Solver,
        'penalty' : Panality,
        "Accuracy": acc,
        "ROC_AUC": roc_auc,
        "MSE": mse,
        "MAE": mae
    }])
    print("================ Logistice Model ================")
    print(f'Logistice accureacy {acc}')
    return model, results_df


In [6]:
def logistic_model_3(X_train, X_test, y_train, y_test):
    # Logistic Regression with elasticnet penalty, saga solver
    # 1. saga + elasticnet + l1_ratio=0.1, C=10
    C_value = 1
    L_1_ratio = 0.9
    Solver = 'saga'
    Panality = 'elasticnet'
    model = LogisticRegression(
    penalty = Panality,
    solver = Solver,
    l1_ratio = L_1_ratio,
    C = C_value,
    max_iter = 10000
    )
    model.fit(X_train, y_train)

    # Predictions
    y_pred_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1
    y_pred_class = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred_class)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    mse = mean_squared_error(y_test, y_pred_prob)
    mae = mean_absolute_error(y_test, y_pred_prob)

    # Store results in DataFrame
    results_df = pd.DataFrame([{
        "C": C_value,
        "l1_ratio": L_1_ratio,
        'solver': Solver,
        'penalty' : Panality,
        "Accuracy": acc,
        "ROC_AUC": roc_auc,
        "MSE": mse,
        "MAE": mae
    }])
    print("================ Logistice Model ================")
    print(f'Logistice accureacy {acc}')
    return model, results_df


In [7]:
def run_experiment(uri='default_path', experiment_name='default_exp',
                   model_name='logistic_model', run_name='default_run',
                   model_func=None, model_args=None, tags=None):
    # Set tracking directory explicitly
    mlflow.set_tracking_uri(uri)
    print("The set tracking uri is ", mlflow.get_tracking_uri())

    client = MlflowClient()

    # ✅ Check if experiment exists
    exp = client.get_experiment_by_name(experiment_name)
    if exp:
        if exp.lifecycle_stage == "deleted":
            print(f"Experiment '{experiment_name}' is deleted. Restoring...")
            client.restore_experiment(exp.experiment_id)
        exp_id = exp.experiment_id
    else:
        # Create new experiment if not found
        exp_id = client.create_experiment(experiment_name)

    # Reuse or set experiment safely
    mlflow.set_experiment(experiment_name=experiment_name)
    get_exp = mlflow.get_experiment(exp_id)
    print("Name:", get_exp.name)
    print("Experiment_id:", get_exp.experiment_id)
    print("Artifact Location:", get_exp.artifact_location)
    print("Tags:", get_exp.tags)
    print("Lifecycle_stage:", get_exp.lifecycle_stage)
    print("Creation timestamp:", get_exp.creation_time)

    with mlflow.start_run(experiment_id=exp_id, run_name=run_name):
        if tags:
            mlflow.set_tags(tags)

        # Train and evaluate model
        model, results_df = model_func(**model_args)

        # Log parameters
        for param in ["C", "l1_ratio", "solver", "penalty"]:
            if param in results_df.columns:
                mlflow.log_param(param, results_df.loc[0, param])

        # Log metrics
        for metric in ["Accuracy", "ROC_AUC", "MSE", "MAE"]:
            if metric in results_df.columns:
                mlflow.log_metric(metric, results_df.loc[0, metric])

        # Log model
        mlflow.sklearn.log_model(model, name=model_name, serialization_format="skops")

        # Log artifacts (optional)
        mlflow.log_artifacts("data/")

        # Print artifact URI
        artifacts_uri = mlflow.get_artifact_uri()
        print("The artifact path is", artifacts_uri)

    mlflow.end_run()

    run = mlflow.last_active_run()
    if run:
        print("Active run id:", run.info.run_id)
        print("Active run name:", run.info.run_name)


In [8]:
def main():
    warnings.filterwarnings("ignore")
    np.random.seed(40)
    # Load data
    X_train, X_test, y_train, y_test = get_cleaned_data()
    # Example usage
    tags = {
    "Work": "Embedded Platform",
    "release.candidate": "RELAY_01",
    "release.version": "1.0.10",
    "dataset": "Cholesterol",
    "experiment.stage": "hyperparameter_tuning",
    "owner": "abhishek",
    "framework": "scikit-learn",
    "model.type": "logistic"
    }
    #uri_path = get_mlflow_uri_path()
    uri_path='http://127.0.0.1:5000'
    run_experiment(uri=uri_path,
                   experiment_name='logistice_11', 
                   model_name='logistice_regressions',
                   run_name='Model_runs', 
                   model_func=logistic_model_1,
                   model_args={
                                "X_train": X_train,
                                "X_test": X_test,
                                "y_train": y_train,
                                "y_test": y_test,
        
                                },
                   tags=tags)
    
    run_experiment(uri=uri_path,
                   experiment_name='logistice_2', 
                   model_name='logistice_regressions',
                   run_name='Model_runs', 
                   model_func=logistic_model_2,
                   model_args={
                                "X_train": X_train,
                                "X_test": X_test,
                                "y_train": y_train,
                                "y_test": y_test,
        
                                },
                   tags=tags)
    
    run_experiment(uri=uri_path,
                   experiment_name='logistice_3', 
                   model_name='logistice_regressions',
                   run_name='Model_runs', 
                   model_func=logistic_model_3,
                   model_args={
                                "X_train": X_train,
                                "X_test": X_test,
                                "y_train": y_train,
                                "y_test": y_test,
        
                                },
                   tags=tags)

In [9]:
if __name__ == "__main__":
    main()

The set tracking uri is  http://127.0.0.1:5000
Name: logistice_11
Experiment_id: 7
Artifact Location: file:C:/Users/abhis/artifacts/7
Tags: {}
Lifecycle_stage: active
Creation timestamp: 1779381628602
================ Logistice Model ================
Logistice accureacy 0.8333333333333334
The artifact path is file:C:/Users/abhis/artifacts/7/743e5bd1cfe749d795288edcd950077c/artifacts
🏃 View run Model_runs at: http://127.0.0.1:5000/#/experiments/7/runs/743e5bd1cfe749d795288edcd950077c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
Active run id: 743e5bd1cfe749d795288edcd950077c
Active run name: Model_runs
The set tracking uri is  http://127.0.0.1:5000
Name: logistice_2
Experiment_id: 8
Artifact Location: file:C:/Users/abhis/artifacts/8
Tags: {}
Lifecycle_stage: active
Creation timestamp: 1779381637485
================ Logistice Model ================
Logistice accureacy 0.8333333333333334
The artifact path is file:C:/Users/abhis/artifacts/8/0a992a7beb1e40e8ae677f566b125920/a